# Assignment 2 — Add Memory to Your Agent



In [1]:
%pip install transformers langchain langchain-community langchain-huggingface ddgs huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# --- LOCAL MODEL: runs offline, no token needed ---
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

llm = HuggingFacePipeline(pipeline=pipeline(
    'text2text-generation', model='google/flan-t5-base', max_new_tokens=256))
print('LLM ready: local flan-t5-base')

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM ready: local flan-t5-base


C:\Users\hp\AppData\Local\Temp\ipykernel_15372\3607214097.py:5: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipeline(


In [3]:
import re
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun

# --- Tool 1: web search (DuckDuckGo) ---
_ddg = DuckDuckGoSearchRun()
_ddg.api_wrapper.backend = 'html'  # real page snippets

def search_tool(query: str) -> str:
    """Search the web and return a short text snippet."""
    try:
        return _ddg.run(query)[:500]
    except Exception as e:
        return f'Search error: {e}'

# --- Tool 2: calculator (safe arithmetic only) ---
def calculator_tool(expr: str) -> str:
    """Evaluate a math expression like '4.4 * 0.05'."""
    if not re.fullmatch(r'[0-9\.\+\-\*\/\(\) ]+', expr):
        return 'Calculator error: invalid expression.'
    try:
        return str(eval(expr))
    except Exception as e:
        return f'Calculator error: {e}'

# Registry the agent will pick from
TOOLS = {'search': search_tool, 'calculator': calculator_tool}
print('Tools available:', list(TOOLS))

Tools available: ['search', 'calculator']


## Add memory

`ConversationBufferMemory` stores the running conversation. Before each new question we read the stored history and prepend it to the prompt, so the model has the earlier context.

In [4]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

# (A plain dict works too — exactly like memory_store in your LangChain notebook:
#   memory_store = {}  ; memory_store['history'] += ... )

REACT = '''You are a helpful agent with tools: search, calculator.
Use the conversation history for context (e.g. 'that number' refers to an earlier result).

Conversation so far:
{history}

Use the format:
Thought: ...
Action: <search|calculator>
Action Input: ...
Observation: ...
Thought: I now know the answer
Final Answer: ...

Question: {question}
'''

def run_agent_with_memory(question, max_steps=5, verbose=True):
    history = memory.load_memory_variables({})['history']
    transcript = REACT.format(history=history or '(none yet)', question=question)
    final = None
    for _ in range(max_steps):
        out = llm.invoke(transcript)
        transcript += out
        if verbose: print(out.strip())
        if 'Final Answer:' in out:
            final = out.split('Final Answer:')[-1].strip(); break
        action = re.search(r'Action:\s*(\w+)', out)
        ainput = re.search(r'Action Input:\s*(.+)', out)
        if not action or not ainput:
            final = out.strip(); break
        result = TOOLS.get(action.group(1).strip(), lambda x: 'Unknown tool')(ainput.group(1).strip())
        transcript += f'\nObservation: {result}\n'
        if verbose: print('Observation:', result, '\n')
    # SAVE this turn into memory so the next call can recall it
    memory.save_context({'input': question}, {'output': final or ''})
    return final

C:\Users\hp\AppData\Local\Temp\ipykernel_15372\2179118003.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


## Demonstrate recall — turn 2 refers back to turn 1:

In [5]:
print('--- Turn 1 ---')
print('ANSWER:', run_agent_with_memory('What is 250 * 4?', verbose=False))

print('\n--- Turn 2 (no number repeated!) ---')
print('ANSWER:', run_agent_with_memory('Now add 100 to that result.', verbose=False))

--- Turn 1 ---
ANSWER: 4

--- Turn 2 (no number repeated!) ---
ANSWER: Thought: I now know the answer.


Inspect what the agent actually stored:

In [6]:
print(memory.load_memory_variables({})['history'])

Human: What is 250 * 4?
AI: 4
Human: Now add 100 to that result.
AI: Thought: I now know the answer.


## Reflection qyestion

Without memory, every query is isolated — *"add 100 to that"* would be meaningless. By loading the `ConversationBufferMemory` into each prompt, the agent keeps **contextual continuity** and can resolve references like *that result* to the earlier `1000`. A plain `dict` does the same job for simple cases; `ConversationBufferMemory` just standardises save/load and integrates with the rest of LangChain.